In [5]:
import pandas as pd
import numpy as np
import json


with open("data/events_records.json", encoding="utf-8") as f:
    raw = json.load(f)

n = len(raw[0])
df_records = pd.DataFrame({
    "event_id":   list(range(1, n + 1)),
    "user_id":    [raw[1][str(i)] for i in range(n)],
    "event_time": [raw[2][str(i)] for i in range(n)],
    "event_type": [raw[3][str(i)] for i in range(n)],
    "page":       [raw[4][str(i)] for i in range(n)],
})


df_lines = pd.read_json("data/events_lines.json", lines=True)


df_events = pd.concat([df_records, df_lines], ignore_index=True)
df_events["event_time"] = pd.to_datetime(df_events["event_time"])

df_events = df_events.sort_values(["user_id", "event_time"]).reset_index(drop=True)

df_events["prev_event_time"] = (
    df_events.groupby("user_id")["event_time"].shift(1)
)
df_events["delta_since_prev"] = (
    df_events["event_time"] - df_events["prev_event_time"]
)

session_timeout = pd.Timedelta(minutes=30)

df_events["is_new_session"] = (
    df_events["delta_since_prev"].isna()
    | (df_events["delta_since_prev"] > session_timeout)
)

df_events["session_id"] = (
    df_events.groupby("user_id")["is_new_session"].cumsum()
)

last_events = (
    df_events
    .sort_values("event_time")
    .groupby(["user_id", "session_id"], as_index=False)
    .tail(1)
    .copy()
)

last_events["ended_with_purchase"] = (
    last_events["event_type"] == "purchase"
).astype(int)

print(last_events[["user_id", "session_id", "event_time",
                   "event_type", "ended_with_purchase"]].head(20))

      user_id  session_id          event_time   event_type  \
251      1007           1 2024-01-01 01:06:07    page_view   
2795     1071           1 2024-01-01 02:19:12     purchase   
2349     1060           1 2024-01-01 02:40:35  add_to_cart   
3139     1079           1 2024-01-01 04:05:58    page_view   
448      1012           1 2024-01-01 04:49:08  add_to_cart   
1012     1027           1 2024-01-01 05:42:38  add_to_cart   
400      1011           1 2024-01-01 05:48:52    page_view   
3067     1077           1 2024-01-01 06:19:23    page_view   
0        1001           1 2024-01-01 06:25:31     purchase   
1163     1031           1 2024-01-01 08:13:30       scroll   
3969     1100           1 2024-01-01 09:26:16    page_view   
1164     1031           2 2024-01-01 10:55:33     purchase   
3752     1095           1 2024-01-01 10:56:28  add_to_cart   
2350     1060           2 2024-01-01 11:12:54    page_view   
2011     1052           1 2024-01-01 12:00:42        click   
1048    

In [6]:
user_session_stats = (
    last_events
    .groupby("user_id")
    .agg(
        sessions_total=("session_id", "count"),
        sessions_with_purchase=("ended_with_purchase", "sum"),
    )
)

user_session_stats["purchase_session_share"] = (
    user_session_stats["sessions_with_purchase"]
    / user_session_stats["sessions_total"]
)

user_session_stats = user_session_stats.sort_values(
    "purchase_session_share", ascending=False
)

print("\nДоля «покупающих» сессий по пользователям (топ-15):")
print(user_session_stats.head(15))


Доля «покупающих» сессий по пользователям (топ-15):
         sessions_total  sessions_with_purchase  purchase_session_share
user_id                                                                
1003                 39                      16                0.410256
1004                 35                      14                0.400000
1058                 45                      16                0.355556
1076                 44                      14                0.318182
1048                 29                       9                0.310345
1014                 39                      12                0.307692
1091                 63                      19                0.301587
1005                 44                      13                0.295455
1094                 34                      10                0.294118
1022                 38                      11                0.289474
1001                 50                      14                0.280000
1066       

In [7]:

meaningful = user_session_stats[user_session_stats["sessions_total"] >= 3]

top_users = meaningful.sort_values(
    ["purchase_session_share", "sessions_total"],
    ascending=[False, False]
)

print("\nПользователи с наибольшей долей 'покупающих' сессий (>= 3 сессий):")
print(top_users.head(10))

top_users.head(10).to_csv("top_purchase_users.csv")


Пользователи с наибольшей долей 'покупающих' сессий (>= 3 сессий):
         sessions_total  sessions_with_purchase  purchase_session_share
user_id                                                                
1003                 39                      16                0.410256
1004                 35                      14                0.400000
1058                 45                      16                0.355556
1076                 44                      14                0.318182
1048                 29                       9                0.310345
1014                 39                      12                0.307692
1091                 63                      19                0.301587
1005                 44                      13                0.295455
1094                 34                      10                0.294118
1022                 38                      11                0.289474
